# Week 4 -- Strategy Reflection
### Imperial College -- Bayesian Optimisation Capstone

---

This notebook reflects on the shift to neural network surrogates and TensorFlow-based gradient optimisation in Week 4.
All answers are written based on results from Weeks 1 to 3 and the strategy decisions made for Week 4.

**Framework this week:** TensorFlow (Sequential API + GradientTape)

---
## 1. Support Vectors and Decision Boundary Points

After three rounds of queries, some observations sit right on the edge between good and poor output regions. These behave like support vectors in an SVM -- they define where the boundary is.

For **F5**, the data points with yields around 800-900 sit at the edge of the good region. The three weekly queries pushed the value from there up to 1192, crossing into clearly better territory. Those boundary points showed where to go next.

For **F7**, the results went 0.679, 0.775, then 1.235. The observations around 0.7-0.8 are right at the boundary between average and good. They told me which direction to keep pushing.

For **F4** it worked the other way. The best result (-24.548 in Week 1) was clearly separated from the region where the search got stuck at -28.564. Recognising that the acquisition function had drifted into a worse attractor earlier would have helped me switch strategy sooner.

The main lesson: observations near the good/poor boundary are the most informative ones. They tell you where the edge is, which is more useful than points that are clearly good or clearly bad.

---
## 2. Using Neural Network Gradients as Directional Signals

A Gaussian process tells you where it is uncertain. A neural network tells you something different -- it can tell you which direction to move to improve the output.

This works through TensorFlow's GradientTape. You declare the input as a variable the tape watches, run the model forward, then ask for the gradient of the output with respect to each input dimension. The result is a score per dimension: large gradient means that dimension has a big effect, near-zero means changing it will not help much.

For **F7** (6 dimensions), this gradient analysis showed that two or three dimensions drove most of the predicted output change. The others were nearly flat. That told me which dimensions to focus the next query on rather than moving all six at once.

For **F8** (8 dimensions), this was the most valuable tool. With only 43 observations across 8 dimensions, the gradient ranking compressed the search. Instead of an 8D problem, I only needed to vary the three high-gradient dimensions. The other five stayed close to their best-known values.

For **F5**, when all four gradients were near zero, the NN was saying the current point is already close to a peak. That was a signal to refine locally rather than move far away.

---
## 3. Framing BBO as a Classification Task: Good vs Bad

Instead of trying to predict the exact output value, you can simplify the problem: label observations in the top 30% as good and the rest as bad, then train a classifier to find the boundary between them.

**Logistic regression** draws a straight line boundary. For F1 where the good region is a small patch in the middle of a mostly near-zero space, a straight line would not capture the shape of the boundary at all.

**SVM with an RBF kernel** can draw curved boundaries that close around the good region. This fits better for most functions here where the good region is a connected patch surrounded by poor results. The risk is overfitting -- with only 12-43 observations per function, the SVM may chase individual good points rather than learning the general boundary shape.

**Neural networks** can draw even more complex boundaries but are most prone to overfitting at this scale. With 33 observations in F7 across 6 dimensions, the NN risks memorising the labels rather than learning the true structure.

The trade-off in all three cases is between being too cautious (missing a good region) and being too aggressive (wasting queries on poor regions). For this project, erring toward classifying uncertain regions as potentially good makes more sense -- missing the optimum entirely is a bigger cost than one wasted query.

---
## 4. Model Choice: Interpretability vs Flexibility

**Linear regression** is the most interpretable -- one coefficient per dimension tells you directly how each input affects the output. But it cannot capture curves or interactions between dimensions, and its gradient is constant everywhere, which gives no useful direction for acquisition.

**Gaussian process with a fixed RBF kernel** sits in the middle. It interpolates smoothly between observations and gives uncertainty estimates, which is useful for acquisition. The limitation is the fixed kernel -- same smoothness everywhere -- and it cannot produce a differentiable surface for gradient-based input optimisation.

**Neural network (TensorFlow Sequential)** is the most flexible. It learns the shape of the response surface without assuming it is smooth or symmetric. The key benefit in Week 4 is that the gradient of the output with respect to inputs is available cheaply through backpropagation, and that gradient directly drives the acquisition in Cell 6.

The costs are real: the NN has no uncertainty estimate (unlike the GP), and with small datasets it can overfit. Both problems are managed here -- Dropout(0.1) and L2 regularisation in each layer reduce overfitting, and the GP is kept as a fallback candidate in the dual acquisition.

The hybrid approach in Week 4 uses both: the NN finds the gradient-guided candidate, the GP finds the uncertainty-guided candidate, and whichever scores higher under the NN wins.

---
## 5. Steepest Gradient Dimensions and How to Use Them

The sensitivity analysis in each notebook ranks input dimensions by how much the model output changes when that dimension changes at the current best point.

For **F8** (8 dimensions, 43 observations), this ranking is essential. If three dimensions show high gradients and five show near-zero, you only need to vary those three in the next query. The others stay fixed near their best-known values. This turns an 8D problem into a 3D one.

For **F5** (4 dimensions), near-zero gradients in all four directions mean the model thinks the current point is close to a local maximum. Any direction is roughly flat. The right response is to refine very locally rather than making a large jump.

For **F4**, which had been stuck, the gradient at the Week 1 best point shows which dimension has the most leverage for escaping. That is more useful than a random new point -- it gives a specific direction based on what the model has learned.

Going into Week 5, the gradient rankings across all eight functions gave a clearer picture of where to focus each query than the beta-tuning approach used in Weeks 2 and 3.

---
## 6. How Well Did the NN Approximate the Decision Boundary?

When observations are labelled good/bad and a neural network is trained on them, the network learns a boundary in the input space. Backpropagation gives access to the gradient of that boundary, which tells you which direction crosses from bad into good.

For **F2** (2D), the best known point is at [1.0, 0.326]. The NN predicts high output in the high-x1, mid-x2 region. The boundary runs diagonally across the 2D space. Compared to the GP, the NN is more willing to predict extreme values in unvisited regions because it has no uncertainty mechanism. That can be useful (clearer gradient direction) or risky (overconfident in regions with no data).

For **F7** (6D), the boundary cannot be visualised, but the gradient at the current best point shows which direction is 'more good'. Backpropagation computes this at any point, including near the boundary between explored good territory and unexplored space.

The main limitation: the boundary the NN learns is only as good as the data it was trained on. With 33 observations in F7, the boundary will shift as more data arrives. It is an approximation that improves over time, not a fixed truth.

---
## 7. Was the Neural Network Worth the Extra Complexity?

It depends on the function.

For **F1 and F2 (2D)**, the GP is probably sufficient on its own. The response surface can be understood directly from the data in two dimensions. The NN adds gradient computation, but the main reason to use it for these functions is consistency -- the same pipeline structure across all eight functions rather than a bespoke approach for low-dimensional ones.

For **F7 and F8 (6D, 8D)**, the NN gives a clear advantage. The GP with a fixed kernel treats all dimensions equally and cannot learn which ones matter. The NN learns asymmetric surfaces and -- most importantly -- provides the gradient sensitivity ranking that makes high-dimensional search practical. For F8, the jump from 7.627 to 9.038 in Week 3 came from a very different region of the space. The NN can interpolate across that kind of jump better than a fixed-kernel GP.

The honest summary: for simple 2D functions the added complexity is marginal. For 6D and 8D functions with limited data, the gradient-based dimension ranking from the NN is the most useful tool available and justifies the extra setup.

---
*Reflection written after Week 4 submission | Imperial College Bayesian Optimisation Capstone*